# Experiment Report Dashboard

This notebook reads saved experiment outputs and compares how each model behaved across experiments. It is designed to work with whatever model names appear in the saved reports, including `ridge`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

def find_task_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == "task-house-prices" and (candidate / "outputs").exists():
            return candidate
        nested = candidate / "courses/01-machine-learning-with-python/tasks/task-house-prices"
        if nested.exists() and (nested / "outputs").exists():
            return nested
    fallback = Path("/Users/maksymponarenko/Documents/ai-engineering-kaggle-portfolio/courses/01-machine-learning-with-python/tasks/task-house-prices")
    if fallback.exists():
        return fallback
    raise FileNotFoundError("Could not locate the task-house-prices project root.")

TASK_DIR = find_task_root()
OUTPUTS_DIR = TASK_DIR / "outputs" / "experiments"

print("Task directory:", TASK_DIR)
print("Outputs directory:", OUTPUTS_DIR)


## Load summary data

If the global summary file exists, we use it. Otherwise we reconstruct a combined table from each saved experiment folder.

In [ ]:
summary_path = OUTPUTS_DIR / "experiment_summary_all_models.csv"

if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
else:
    frames = []
    for experiment_dir in sorted(path for path in OUTPUTS_DIR.iterdir() if path.is_dir()):
        comparison_path = experiment_dir / "model_comparison.csv"
        definition_path = experiment_dir / "experiment_definition.json"
        if not comparison_path.exists():
            continue
        frame = pd.read_csv(comparison_path)
        frame["experiment_name"] = experiment_dir.name
        frame["experiment_dir"] = str(experiment_dir)
        frames.append(frame)
    summary_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if summary_df.empty:
    raise ValueError("No saved experiment reports were found in outputs/experiments.")

if "model_name" not in summary_df.columns and "model" in summary_df.columns:
    summary_df = summary_df.rename(columns={"model": "model_name"})

summary_df = summary_df.sort_values(["experiment_name", "rmse_mean", "model_name"]).reset_index(drop=True)
available_models = summary_df["model_name"].dropna().drop_duplicates().tolist()

print("Available models:", available_models)
display(summary_df.head(20))

## Best model per experiment

This is a quick summary, but the main point of the dashboard is still the full comparison across all models.

In [ ]:
best_df = summary_df.sort_values("rmse_mean").groupby("experiment_name", as_index=False).first()
display(best_df[["experiment_name", "model_name", "rmse_mean", "mae_mean", "r2_mean"]])

## Model coverage check

This table helps detect mixed old/new saved reports. If a model is missing for an experiment, the report was likely saved before that model was included in the save configuration.

In [ ]:
expected_models = sorted(available_models)
coverage_rows = []
for experiment_name, group in summary_df.groupby("experiment_name"):
    present_models = sorted(group["model_name"].dropna().unique().tolist())
    missing_models = [model for model in expected_models if model not in present_models]
    coverage_rows.append(
        {
            "experiment_name": experiment_name,
            "present_models": ", ".join(present_models),
            "missing_models": ", ".join(missing_models) if missing_models else "",
            "model_count": len(present_models),
        }
    )
coverage_df = pd.DataFrame(coverage_rows).sort_values(["model_count", "experiment_name"], ascending=[True, True])
display(coverage_df)

incomplete_reports = coverage_df.loc[coverage_df["missing_models"] != ""].copy()
if not incomplete_reports.empty:
    print("Some saved experiments are missing models. Re-save those experiments from feature_experiment_lab.ipynb with SAVE_EXPERIMENT_OUTPUTS=True.")
    display(incomplete_reports)
else:
    print("All saved experiments contain the same set of models.")

## Pivot tables

Each pivot table automatically includes every model found in the saved reports.

In [ ]:
rmse_pivot = summary_df.pivot_table(index="experiment_name", columns="model_name", values="rmse_mean", aggfunc="first")
mae_pivot = summary_df.pivot_table(index="experiment_name", columns="model_name", values="mae_mean", aggfunc="first")
r2_pivot = summary_df.pivot_table(index="experiment_name", columns="model_name", values="r2_mean", aggfunc="first")

display(rmse_pivot)
display(mae_pivot)
display(r2_pivot)

## Heatmaps

The heatmaps are dynamic too, so `ridge` will show up automatically if it exists in the reports.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, max(4, len(rmse_pivot) * 0.6)))
sns.heatmap(rmse_pivot, annot=True, fmt=".0f", cmap="YlOrRd_r", ax=axes[0])
axes[0].set_title("RMSE by experiment and model")
sns.heatmap(mae_pivot, annot=True, fmt=".0f", cmap="YlOrBr_r", ax=axes[1])
axes[1].set_title("MAE by experiment and model")
sns.heatmap(r2_pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[2])
axes[2].set_title("R2 by experiment and model")
plt.tight_layout()

## Model traces across experiments

This view is helpful when you want to track one model at a time and see whether a new feature idea helped or hurt.

In [ ]:
metric_name = "r2_mean"
plot_df = summary_df.copy()

plt.figure(figsize=(14, 6))
sns.lineplot(data=plot_df, x="experiment_name", y=metric_name, hue="model_name", marker="o")
plt.xticks(rotation=45, ha="right")
plt.title(f"{metric_name} across experiments")
plt.tight_layout()

## Drill-down: one model

Choose a model name from `available_models` and inspect only its history.

In [ ]:
SELECTED_MODEL = available_models[0]
selected_model_df = summary_df.loc[summary_df["model_name"] == SELECTED_MODEL].copy()
display(selected_model_df)

plt.figure(figsize=(12, 5))
sns.barplot(data=selected_model_df, x="experiment_name", y="r2_mean")
plt.xticks(rotation=45, ha="right")
plt.title(f"R2 for {SELECTED_MODEL} across experiments")
plt.tight_layout()

## Drill-down: one experiment

Choose an experiment name and inspect every model that was saved for it.

In [ ]:
SELECTED_EXPERIMENT = summary_df["experiment_name"].iloc[0]
selected_experiment_df = summary_df.loc[summary_df["experiment_name"] == SELECTED_EXPERIMENT].copy()
display(selected_experiment_df.sort_values("rmse_mean"))

plt.figure(figsize=(10, 5))
sns.barplot(data=selected_experiment_df, x="model_name", y="r2_mean")
plt.title(f"R2 by model for {SELECTED_EXPERIMENT}")
plt.tight_layout()